# Reproducing Komori et al. (2026) from DANDI:001425: IC count, window, VIF and SHAP

**Paper.** Komori Y, Mizusaki K, Tomita K, Yoneda T, Matoba K, Morita T (2026). Mesoscopic cortical
activities associated with pupil-linked perceptions inferred via explainable machine learning.
bioRxiv, [doi:10.64898/2026.05.27.726399](https://doi.org/10.64898/2026.05.27.726399). The claims
file (`claims/komori-2026.yaml`) was written from the preprint posted June 1, 2026, with its
Methods, legends and Supplementary Figures 3 to 6.

**Dandiset.** [DANDI:001425](https://dandiarchive.org/dandiset/001425/0.250705.0947), version
0.250705.0947 ("BraiDyn-BC: mouse cued-lever-pull task dataset", Kondo et al. 2025). The paper
cites the dandiset without a version. Two versions are published, 0.250514.0602 (1814 assets) and
0.250705.0947 (1838 assets); the most recent is pinned, as the protocol asks when no version is
cited. The paper was posted in 2026, after both, so either could be the one it read. The data check
shows that 6 of the 9 files read here are the same assets in both versions; the other 3 were
replaced in the later version, and the earlier ones were not compared.
The paper uses only the resting-state recordings of this dandiset; its own recordings
(somatosensory and visual stimulation) are not deposited.

**What is tested.** The six resting-state claims. The paper's pipeline is rebuilt from its
Methods: hemodynamic correction, SVD and JADE ICA of the cortical images, pupil labels around the
dilation peak of each 40 s epoch, a GRU decoder with five-fold cross-validation and null models
trained on labels shuffled within epochs. The open details R1 to R10 of the claims file's opening
comment are followed as it names them; where several readings are named, the notebook runs the
main one and as many others as time allows, and says which were not run.

This notebook tests the IC count and window, the VIF and the SHAP claims, and runs one costly
variant of rest-decoding-above-null (R4 b, every frame of the epoch a sample). The decoding claims
are in `reproduce_komori_2026_decoding.ipynb`; the loading and decomposition here are the same.

| Claim | Notebook | Status |
|---|---|---|
| rest-decoding-above-null | decoding | tested (one variant, R4 b, in the features notebook) |
| rest-loss-accuracy-vs-null | decoding | tested |
| rest-needs-0-1-hz | decoding | tested |
| rest-ic-count-and-window | features | tested |
| rest-vif-below-10 | features | tested |
| rest-shap-medial-anterior-first | features | tested |
| fig1-*, fig2-*, fig3-*, fig4-shap-*, fig4-mapped-shap-regions, supp2-*, supp4-*, supp8-*, supp10-* (18 claims) | none | blocked: the stimulation recordings are not deposited |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

import pupilwf as pw

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
assets = pw.list_assets()
rows = pw.resting_table(assets)
chosen = pw.choose_sessions(rows)          # R1 (a): three mice, first three resting sessions each
chosen_rows = [r for mouse in chosen for r in mouse]
sessions = pw.load_sessions(chosen_rows)
print(f"streamed and reduced {len(sessions)} sessions from DANDI:{pw.DANDISET} version {pw.VERSION} "
      f"in {time.time() - started:.0f} s")

In [ ]:
t0 = time.time()
D = pw.decompose(sessions, hemo="fit_hp_subtract_raw")       # R2 (a), main reading
comp = pw.compactness(D["maps"], D["mask"])
# features for VIF-based selection: every frame of the 13 epochs, 0.1 Hz low-pass (R6)
def vif_frames(ics, cutoff=0.1):
    out = []
    for s, ic in zip(sessions, ics):
        a, b = pw.epochs_of(s["t"])[0][0], pw.epochs_of(s["t"])[-1][1]
        out.append(pw.normalise_features(ic, cutoff)[a:b])
    return np.vstack(out)
F01 = vif_frames(D["ics"])
keep10, removed10 = pw.select_ics(F01, comp, 10, "kept")
print(f"decomposition in {time.time() - t0:.0f} s; kept ICs {keep10}")

## Data check

The full data check is in the decoding notebook; this repeats what this notebook uses.

In [ ]:
print("Common cortical grid points (atlas grid, step", pw.GRID_STEP, "atlas pixels):", int(D["mask"].sum()))
print("SVD components available:", D["n_svd"], "(paper: 400; the corrected data of each recording have rank 34)")
print("Cumulative variance at 10, 30, 200 components:", np.round(D["cum_var"][[9, 29, min(199, D['n_svd'] - 1)]], 4),
      "(paper: > 99.5% at 200)")
print("Frames used to estimate the decomposition (every fifth 40 s epoch):", D["n_sel_frames"])
print("IC compactness (share of squared weight in the largest supra-half-maximum region):")
print(pd.Series(comp).round(3).to_string())
v30 = pw.vif(F01)
print("VIF of all 30 ICs (0.1 Hz features): max", round(float(v30.max()), 2))
print("Kept for 10 ICs (reading a):", keep10, "removed for VIF:", removed10,
      "max VIF of the kept:", round(float(pw.vif(F01[:, keep10]).max()), 2))
roi = sessions[0]["roi_atlas"]
names = sessions[0]["roi_names"]
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for ax, j in zip(axes.flat, keep10):
    m = D["maps"][j]
    lim = np.nanmax(np.abs(m))
    ax.imshow(m, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.contour(roi.any(axis=0), [0.5], colors="k", linewidths=0.5)
    ax.set_title(f"IC {j}  comp {comp[j]:.2f}", fontsize=9)
    ax.axis("off")
plt.suptitle("Spatial maps of the 10 kept ICs on the common atlas grid (colour limits: +/- max |weight|)")
plt.tight_layout()
plt.show()

In [ ]:
ds_main = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1)
lab = pd.DataFrame(ds_main["info"])
print("Epochs per recording:", lab.groupby("rec").size().unique(), "; samples per epoch window:", sorted(lab.n.unique()))
print("Positive frames per epoch window (of", lab.n.max(), "), by recording:")
print(lab.groupby(["subject", "day"]).n_pos.describe()[["mean", "min", "50%", "max"]].round(1).to_string())
print("Epochs whose window has no positive frame:", int((lab.n_pos == 0).sum()), "of", len(lab),
      "; all frames positive:", int((lab.n_pos == lab.n).sum()))
print("Samples:", ds_main["X"].shape, "positive share", round(float(ds_main["y"].mean()), 3))
print("Feature range:", float(ds_main["X"].min()), float(ds_main["X"].max()))
lab5 = pd.DataFrame(pw.make_dataset(sessions, D["ics"], keep10, 0.1, selection="rule")["info"])
print("Epochs kept by the selection rule R5 (b):", int(lab5.keep.sum()), "of", len(lab5))

## rest-ic-count-and-window

> For resting data, decoding with 10 ICs and a 1.5 s feature window gave the highest median AUC
> among the IC counts (7, 10, 13) and window lengths (0.5 to 10 s) tested, though no comparison was
> significant.

IC counts under reading (a), the number kept, and (b), the number before VIF pruning; windows of
0.5, 1.0, 1.5, 2.0, 3.0, 5.0 and 10.0 s (15 to 300 frames) with 10 ICs. Main reading combination
otherwise, at this claim's seed.

In [ ]:
SEED4 = pw.seed_of("rest-ic-count-and-window")
ic_sets = {}
for reading, rd in (("a", "kept"), ("b", "initial")):
    for n in (7, 10, 13):
        ic_sets[(reading, n)] = pw.select_ics(F01, comp, n, rd)
ic_rows, ic_cv, by_set = [], {}, {}
for (reading, n), (keep, removed) in ic_sets.items():
    key = tuple(sorted(int(k) for k in keep))
    if key not in by_set:      # readings (a) and (b) give the same set when no IC is removed for VIF
        by_set[key] = pw.run_cv(pw.make_dataset(sessions, D["ics"], keep, cutoff=0.1), SEED4)
    cv = by_set[key]
    ic_cv[(reading, n)] = cv
    ic_rows.append(dict(reading=reading, n_ics=n, kept=len(keep), removed_for_vif=len(removed),
                        median_auc=float(cv.model_val_auc.median()), mean_auc=float(cv.model_val_auc.mean()),
                        min_auc=float(cv.model_val_auc.min()), max_auc=float(cv.model_val_auc.max())))
it = pd.DataFrame(ic_rows)
print(it.round(4).to_string(index=False))
for reading in ("a", "b"):
    ref = ic_cv[(reading, 10)].model_val_auc
    ps = [stats.ranksums(ref, ic_cv[(reading, n)].model_val_auc).pvalue for n in (7, 13)]
    sub = it[it.reading == reading]
    print(f"reading {reading}: highest median at {int(sub.loc[sub.median_auc.idxmax(), 'n_ics'])} ICs; "
          f"Wilcoxon rank-sum 10 against 7 and 13, Holm:", np.round(multipletests(ps, method='holm')[1], 4))

In [ ]:
win_rows, win_cv = [], {}
for w_s in (0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0):
    w = int(round(w_s * pw.FS)) + (1 if w_s == 1.5 else 0)   # 1.5 s is the paper's 46 frames
    t0 = time.time()
    ds = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1, window=w)
    cv = ic_cv[("a", 10)] if w == pw.WINDOW else pw.run_cv(ds, SEED4)
    win_cv[w_s] = cv
    win_rows.append(dict(window_s=w_s, frames=w, samples=len(ds["y"]), median_auc=float(cv.model_val_auc.median()),
                         mean_auc=float(cv.model_val_auc.mean()), null_mean_auc=float(cv.null_val_auc.mean()),
                         s=round(time.time() - t0)))
wt = pd.DataFrame(win_rows)
print(wt.round(4).to_string(index=False))
ref = win_cv[1.5].model_val_auc
others = [w for w in win_cv if w != 1.5]
ps = [stats.ranksums(ref, win_cv[w].model_val_auc).pvalue for w in others]
print("highest median AUC at window", float(wt.loc[wt.median_auc.idxmax(), "window_s"]), "s")
print("Wilcoxon rank-sum 1.5 s against", others, "Holm:", np.round(multipletests(ps, method="holm")[1], 4))

**Result: partially reproduced (resting on assumptions).** Among the IC counts, 10 ICs have the
highest median validation AUC (0.723; 13 ICs 0.721, 7 ICs 0.695). Among the windows, 1.0 s is
highest (0.730), just above 1.5 s (0.723). As in the paper, no comparison with 10 ICs or with 1.5 s
is significant after Holm correction (all p = 1.0). The differences are small, and at these
settings the models are not above their null models (mean null AUC 0.694 to 0.711). Readings (a)
and (b) of the IC count give the same IC sets, as no IC is removed for VIF (the largest VIF among
all 30 ICs is 5.0).

## rest-vif-below-10

> With 10 ICs the resting-state features have a maximum variance inflation factor of 9.0 across
> features and time frames, below the VIF < 10 bound; 7 ICs gave a significantly lower maximum VIF,
> 13 and 15 ICs did not differ from 10. (Expected 9.0 +/- 0.9.)

VIF within each time frame of the validation samples, maximum over features and frames, per model
(fold); the folds are those of rest-decoding-above-null. IC sets as in rest-ic-count-and-window,
both readings, plus 15 ICs. Deterministic given the folds.

In [ ]:
SEED1 = pw.seed_of("rest-decoding-above-null")
def max_vif_per_fold(keep):
    ds = pw.make_dataset(sessions, D["ics"], keep, cutoff=0.1)
    units = ds["unit"]
    by_mouse = {}
    for u, m in zip(units, ds["mouse"]):
        by_mouse.setdefault(m, set()).add(int(u))
    fold = pw.folds_by_mouse({m: sorted(v) for m, v in sorted(by_mouse.items())}, SEED1)
    fs = np.array([fold[int(u)] for u in units])
    out = []
    for f in range(5):
        Xv = ds["X"][fs == f]
        out.append(max(float(pw.vif(Xv[:, k, :]).max()) for k in range(Xv.shape[1])))
    return np.array(out)
vif_rows, vif_vals = [], {}
for reading, rd in (("a", "kept"), ("b", "initial")):
    for n in (7, 10, 13, 15):
        keep, removed = pw.select_ics(F01, comp, n, rd)
        v = max_vif_per_fold(keep)
        vif_vals[(reading, n)] = v
        vif_rows.append(dict(reading=reading, n_ics=n, kept=len(keep), max_over_models=v.max(), median=np.median(v),
                             per_model=np.round(v, 2).tolist()))
print(pd.DataFrame(vif_rows).round(3).to_string(index=False))
for reading in ("a", "b"):
    ref = vif_vals[(reading, 10)]
    ps, dirs = [], []
    for n in (7, 13, 15):
        ps.append(stats.mannwhitneyu(vif_vals[(reading, n)], ref, alternative="two-sided").pvalue)
        dirs.append(float(np.median(vif_vals[(reading, n)]) - np.median(ref)))
    print(f"reading {reading}: Mann-Whitney U against 10 ICs (7, 13, 15), Holm:",
          np.round(multipletests(ps, method="holm")[1], 4), "; median difference", np.round(dirs, 3))

**Result: not reproduced (resting on assumptions).** With 10 ICs the maximum VIF over features and
frames of the validation samples is 6.61 over the five models (per model 3.48 to 6.61): below the
bound of 10, but outside 9.0 +/- 0.9. 7 ICs give a lower maximum (median 3.13 against 5.30), not
significantly (Holm p = 0.064); 13 ICs do not differ from 10 (p = 0.064), and 15 ICs give a
significantly higher maximum (median 9.05, p = 0.024), where the paper finds no difference. The two
readings of the IC count give the same sets.

## rest-shap-medial-anterior-first

> In the resting-state model, SHAP values of an IC in the medial anterior cortex (IC7) rise before
> those of the medial posterior region (IC6) and the secondary motor cortex (IC4).

The models of rest-decoding-above-null are fitted again here (same data, folds and seeds).
Attribution: integrated gradients from the mean validation input (see the helper; Deep SHAP's
DeepExplainer does not support PyTorch GRU layers), for the first validation sample of each
validation epoch whose prediction is positive: at 0.5 (Methods reading) or at the model's Youden
threshold (code reading). SHAP is averaged over these samples per model, and the 5 models are
compared with the 5 null models per IC and 0.2 s window by paired t-test, uncorrected. The three ICs
are matched by their maps: medial anterior (largest share of positive weight in a band of 0.5 mm
on each side of the midline anterior to RSP, excluding RSP), RSP (RSPd and RSPagl) and MOs.

In [ ]:
from sklearn.metrics import roc_curve
ds = ds_main
cv_s, models_s = pw.run_cv(ds, SEED1, keep_models=True)
print("Refitted models, validation AUC:", np.round(cv_s.model_val_auc, 3).tolist(),
      "null:", np.round(cv_s.null_val_auc, 3).tolist())
def attributions(reading):
    per_model = {"model": [], "null": []}
    counts = []
    for net, nnet, va in models_s:
        Xv, yv, uv = ds["X"][va], ds["y"][va], ds["unit"][va]
        base = Xv.mean(axis=0)
        for kind, m in (("model", net), ("null", nnet)):
            p = pw.predict(m, Xv)
            if reading == "methods":
                thr = 0.5
            else:
                fpr, tpr, th = roc_curve(yv, p)
                thr = th[np.argmax(tpr - fpr)]
            idx = [np.flatnonzero((uv == u) & (p >= thr))[0] for u in np.unique(uv) if ((uv == u) & (p >= thr)).any()]
            counts.append(dict(kind=kind, n=len(idx), thr=float(thr)))
            if not idx:
                per_model[kind].append(np.full(Xv.shape[1:], np.nan))
                continue
            sh = pw.integrated_gradients(m, Xv[idx], base)
            per_model[kind].append(sh.mean(axis=0))   # frames x ICs
    return np.array(per_model["model"]), np.array(per_model["null"]), pd.DataFrame(counts)
shap = {r: attributions(r) for r in ("methods", "code")}
for r, (sm, sn, cnt) in shap.items():
    print(r, "samples explained per model:", cnt[cnt.kind == "model"].n.tolist(), "null:", cnt[cnt.kind == "null"].n.tolist())

In [ ]:
# Match the three ICs by their maps.
roi = sessions[0]["roi_atlas"]
names = sessions[0]["roi_names"]
def region(prefixes):
    return np.any([roi[i] for i, n in enumerate(names) if any(n.startswith(p) for p in prefixes)], axis=0)
rsp, mos = region(["RSP"]), region(["MOs"])
left = region([n for n in names if n.endswith("_l")])
right = region([n for n in names if n.endswith("_r")])
cols = np.arange(roi.shape[2])
xl = (left * cols).sum() / left.sum()
xr = (right * cols).sum() / right.sum()
rows_idx = np.arange(roi.shape[1])
rsp_rows = rows_idx[rsp.any(axis=1)]
mid = (xl + xr) / 2
# grid pixel size: atlas pixels per data pixel (affine scale) times the data grid spacing
um_per_grid = pw.GRID_STEP * np.sqrt(abs(np.linalg.det(sessions[0]["affine"][:, :2]))) * 18.18
band = np.abs(cols - mid)[None, :] * um_per_grid <= 500
ant_dir = -1 if (region(["MOs"]).any(axis=1) * rows_idx).sum() / mos.sum() < rsp_rows.mean() else 1
anterior = (rows_idx[:, None] < rsp_rows.min()) if ant_dir == -1 else (rows_idx[:, None] > rsp_rows.max())
cortex = roi.any(axis=0)
medant = band & anterior & cortex & ~rsp
def share(j, area):
    m = np.nan_to_num(D["maps"][j])
    pk = m.flat[np.argmax(np.abs(m))]
    m = np.clip(m * np.sign(pk), 0, None) ** 2
    return float(m[area].sum() / m.sum())
sh_tab = pd.DataFrame([{"ic": j, "medial_anterior": share(j, medant), "RSP": share(j, rsp), "MOs": share(j, mos)} for j in keep10])
print(f"midline at grid column {mid:.1f}; {um_per_grid:.1f} um per grid point; medial anterior band {int(medant.sum())} grid points")
print(sh_tab.round(3).to_string(index=False))
ic7 = int(sh_tab.loc[sh_tab.medial_anterior.idxmax(), "ic"])
rest_tab = sh_tab[sh_tab.ic != ic7]
ic6 = int(rest_tab.loc[rest_tab.RSP.idxmax(), "ic"])
ic4 = int(rest_tab[rest_tab.ic != ic6].loc[lambda d: d.MOs.idxmax(), "ic"])
match = {"medial anterior (IC7)": ic7, "RSP (IC6)": ic6, "MOs (IC4)": ic4}
print("matched:", match)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, (k, j) in zip(axes, match.items()):
    m = D["maps"][j]
    lim = np.nanmax(np.abs(m))
    ax.imshow(m, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.contour(cortex, [0.5], colors="k", linewidths=0.5)
    ax.contour(medant, [0.5], colors="g", linewidths=0.8)
    ax.set_title(f"{k}: IC {j}", fontsize=9)
    ax.axis("off")
plt.suptitle("Matched ICs (green: medial anterior band); colour limits +/- max |weight|")
plt.tight_layout()
plt.show()

In [ ]:
edges = np.round(np.arange(-1.5, 0.0001, 0.2), 1)
t_frames = (np.arange(pw.WINDOW) - (pw.WINDOW - 1)) / pw.FS          # -1.5 ... 0 s
wins = [(-1.5, -1.3), (-1.3, -1.1), (-1.1, -0.9), (-0.9, -0.7), (-0.7, -0.5), (-0.5, -0.3), (-0.3, -0.1), (-0.1, 0.0)]
pos = {j: keep10.index(j) for j in match.values()}
out = []
for r, (sm, sn, cnt) in shap.items():
    for k, j in match.items():
        c = pos[j]
        onset = None
        rowp = {}
        for (a, b) in wins:
            sel = (t_frames >= a - 1e-9) & ((t_frames < b - 1e-9) if b < 0 else (t_frames <= b + 1e-9))
            x, y = np.nanmean(sm[:, sel, c], axis=1), np.nanmean(sn[:, sel, c], axis=1)
            ok = np.isfinite(x) & np.isfinite(y)          # a model with no positive prediction has no SHAP
            t, p = stats.ttest_rel(x[ok], y[ok]) if ok.sum() >= 3 else (np.nan, np.nan)
            rowp[f"{a}..{b}"] = round(float(p), 3) if np.isfinite(p) else None
            if onset is None and np.isfinite(p) and p < 0.05 and x.mean() > y.mean():
                onset = a
        mean_curve = np.nanmean(sm[:, :, c], axis=0)
        out.append(dict(reading=r, ic=k, pairs=int(ok.sum()), onset_a=onset, peak_b=float(t_frames[np.argmax(mean_curve)]),
                        peak_value=float(mean_curve.max()), **rowp))
st = pd.DataFrame(out)
print("p-values per window (model against null, paired t over 5 models), onset (a) = first window with p < 0.05 "
      "and model above null; peak (b) = time of the maximum of the mean SHAP over models")
print(st.to_string(index=False))
for r in ("methods", "code"):
    s = st[st.reading == r].set_index("ic")
    for col in ("onset_a", "peak_b"):
        v = s[col]
        a = v["medial anterior (IC7)"]
        prec = [o for o in ("RSP (IC6)", "MOs (IC4)") if a is not None and pd.notna(a) and pd.notna(v[o]) and a < v[o]]
        print(f"{r}, {col}: medial anterior {a}, RSP {v['RSP (IC6)']}, MOs {v['MOs (IC4)']}; precedes: {prec}")

**Result: not reproduced (resting on assumptions).** The matched ICs are 0 (medial anterior band,
share 0.44, a broad map that also covers RSP, 0.41, and MOs, 0.53), 25 (RSP, 0.45) and 13 (MOs,
0.77). Under the Methods reading (prediction at 0.5) no IC has a window where the models' SHAP
exceeds the null models' (smallest p 0.21), and the mean SHAP of all three peaks at 0.0 s. Under the
code reading (Youden threshold) only the MOs IC is above null, in the -0.9 to -0.7 s window
(p = 0.045); the medial anterior IC never is (smallest p 0.45), and all three peak at 0.0 s. No
reading shows the medial anterior IC rising first. The refitted models are not above their null
models (mean validation AUC 0.686 against 0.708), so these attributions explain a decoder that does
not beat chance.

## rest-decoding-above-null, reading R4 b

The costly variant of the decoding claim: every frame of every epoch is a sample, positive only
inside the window around the peak (the claims file's R4 reading b), at the decoding claim's seed.

In [ ]:
t0 = time.time()
ds_b = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1, sample_reading="epoch")
cv_b = pw.run_cv(ds_b, SEED1)
print("samples", len(ds_b["y"]), "positive share", round(float(ds_b["y"].mean()), 4))
print(cv_b[["fold", "model_val_auc", "null_val_auc", "model_val_loss", "null_val_loss", "model_val_acc", "null_val_acc"]].round(3).to_string(index=False))
for m in ("val_auc", "val_loss", "val_acc"):
    r = pw.paired(cv_b, m)
    print(f"{m}: model {r['model']:.3f}, null {r['null']:.3f}, t = {r['t']:.2f}, p = {r['p']:.3g}")
print(f"({time.time() - t0:.0f} s; notebook {time.time() - started:.0f} s)")

**Result: partially reproduced (resting on assumptions), the outcome of rest-decoding-above-null.**
With every frame of an epoch a sample (R4 b; 140,400 samples, 4.1% positive), the models reach a
validation AUC of 0.719 against 0.655 for the null models (paired t = 5.90, p = 0.0041), higher in
all five folds (by 0.023 to 0.084). The effect is significant, but the mean AUC is below the
expected 0.82 +/- 0.082 (0.738 to 0.902), which the criterion calls partial. The five-seed check of
this reading was not run (each run takes about 8.5 minutes); it rests on the claim's seed alone.
For rest-loss-accuracy-vs-null this reading does not reproduce: the loss is 0.161 against 0.166
(p = 0.055) and the accuracy is 0.959 for both, the share of negative labels.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| rest-decoding-above-null | partially reproduced (assumed; R4 b reading only, one seed) |
| rest-loss-accuracy-vs-null | not reproduced (assumed) |
| rest-needs-0-1-hz | not reproduced (assumed) |
| rest-ic-count-and-window | partially reproduced (assumed) |
| rest-vif-below-10 | not reproduced (assumed) |
| rest-shap-medial-anterior-first | not reproduced (assumed) |
| 18 claims on the paper's stimulation recordings | blocked: data missing |

**How the analysis differs from the paper's.** The paper does not say which three mice or which
resting sessions it used; the first three mice recorded, with their first three resting sessions,
are used. The ICs it kept "by visual inspection" are replaced by a repeatable rule (the most compact
maps). Frames are binned 2 x 2 (144 x 144) and the decomposition is made on a common atlas grid; the
SVD keeps 306 components, the rank of the corrected data, in place of 400. The GRU is written in
PyTorch with Keras' defaults, and SHAP is computed by integrated gradients from the mean input in
place of DeepExplainer. Failed pupil detections are replaced. Of the readings the claims file names,
the main combination and one change at a time were run, not every combination; the seed check was
run for the main reading of the decoding claims only. The JADE decomposition is not identical
across software environments (a run of the same code in another environment kept other ICs and gave
other AUCs, with the same outcomes except for the R4 b reading, which was not significant there);
the numbers here are those of this notebook's run.

**What was found about the dataset.** The files hold everything the resting analysis needs: frames
of both channels, pupil diameter and keypoints on the imaging clock, atlas ROIs and the atlas
transform. The excitation wavelengths of the two imaging planes are swapped in the metadata. The
deposited pupil diameter holds unflagged failed detections (values up to 2.9e21 pixels) in two of
the nine sessions read. Six of the 98 resting files hold no behaviour and so no pupil.